# Batch 5 — E26, E27, E28, E06 (validation only)

**Runtime:** Runtime → Change runtime type → **L4 GPU**. Runtime → Run all; about **1 h 45 min – 2 h 15 min**. If Colab disconnects,
Run all again: finished runs are skipped.

Every change is made on top of the parent **E21** and judged against the batch 3 E21 run (per-horizon best epoch, seeds 20261015/16/17).
A 2-epoch check first confirms the code still trains E21 exactly as in batch 3.

| Run | Change vs E21 |
|---|---|
| E26 | + 9 volatility and range inputs (Parkinson, Garman–Klass, Rogers–Satchell 20-day; realised 5/10/60-day; overnight gap; intraday return; daily range) → 41 inputs |
| E27 | Look-back 30 days instead of 60 |
| E28 | Random input masking in training: each input of each window hidden with probability 0.15 |
| E06 | Random context length in training: keep the last m days, m ~ U[20, 60], per batch |
| XGB-E26 | XGBoost with the same 20 extra inputs (reference only) |

Rules and expectations: `claude/batches-4-7-design-2026-10-16.md`. Outputs in `MyDrive/Senior/results/batch5/`.

In [ ]:
!nvidia-smi -L

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT_DIR = f'{SENIOR}/prices', f'{SENIOR}/results'

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, os, subprocess, sys
for f in ['scripts/posthoc_eval.py', 'scripts/select_final.py', 'scripts/factors_tabular.py', 'src/alpha158.py', 'src/features_vol.py']:
    assert os.path.exists(f), f'stop: push the batch 4-7 code first ({f} missing)'
for f in ['scripts/baselines_deep.py', 'scripts/posthoc_eval.py', 'src/multiscale_patchtst.py']:
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
FOLDS = [('2022H2', '2022-07-01', '2022-12-31'), ('2023H1', '2023-01-01', '2023-06-30'),
         ('2023H2', '2023-07-01', '2023-12-31'), ('2024H1', '2024-01-01', '2024-06-30')]
B3, B4, B5, B6, B7 = (f'{OUT_DIR}/batch{i}' for i in (3, 4, 5, 6, 7))
E21_JSON = f'{B3}/e21_stacked.json'
need = [E21_JSON, E21_JSON.replace('.json', '_preds.csv.gz'), f'{OUT_DIR}/xgb_e03.json', f'{OUT_DIR}/xgb_e03_preds.csv.gz']
need += [f'{B3}/e22/{f}_deep{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += [f'{OUT_DIR}/e05/{f}_xgb{s}' for f, _, _ in FOLDS for s in ('.json', '_preds.csv.gz')]
need += []
for f in need:
    assert os.path.exists(f), f'stop: missing {f}'
for d in [B5]:
    os.makedirs(d, exist_ok=True)
print('earlier files found; batch 5 outputs go to', B5)

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# Tests; the notebook stops here if any fails (about 10 minutes)
for t in ['tests/test_pipeline.py', 'tests/test_no_lookahead.py', 'tests/test_new_features.py', 'tests/test_train_options.py', 'tests/test_stopping.py', 'tests/test_multiscale_patchtst.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-1500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
E21 = ['--models', 'multiscale_sector', '--news', 'data/derived/sentiment_scores.csv', '--extra-features', 'e03',
       '--lr', '1e-4', '--dropout', '0.2', '--max-epochs', '30', '--patience', '10', '--cls-only', '--stop-rule', 'every',
       '--no-revin']
def done(path, n=3):
    try:
        return len(json.load(open(path))['models']['multiscale_sector']['runs']) == n
    except Exception:
        return False
def run(label, args, out, n=3):
    if done(out, n):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    subprocess.run(['python', 'scripts/baselines_deep.py', *args, '--out', out], check=True)
def opt(args, **kw):
    a = list(args)
    for k, v in kw.items():
        k = '--' + k.replace('_', '-')
        if k in a: a[a.index(k) + 1] = str(v)
        else: a += [k, str(v)]
    return a

In [ ]:
# Reproducibility check (about 2 min): 2 epochs of E21, seed 20261015, must match batch 3's first 2 epochs exactly
subprocess.run(['python', 'scripts/baselines_deep.py', *opt(E21, max_epochs=2), '--seeds', '20261015', '--out', '/content/repro.json'],
               check=True, capture_output=True)
new = json.load(open('/content/repro.json'))['models']['multiscale_sector']['runs'][0]['history']
old = [r for r in json.load(open(E21_JSON))['models']['multiscale_sector']['runs'] if r['seed'] == 20261015][0]['history'][:2]
d = max(abs(a - b) for u, v in zip(old, new) for a, b in zip(u['es_per_horizon'], v['es_per_horizon']))
print('epochs 1-2 inner hold-out, batch 3 vs now:', [u['es_per_horizon'] for u in old], [v['es_per_horizon'] for v in new])
print('REPRODUCED EXACTLY' if d == 0 else f'NOTE: max difference {d:.4f} - tell Claude (the batch is still valid against a rerun control)')

In [ ]:
run('E26 volatility and range inputs', opt(E21, extra_features='e03_e26'), f'{B5}/e26_vol_inputs.json')

In [ ]:
run('E27 look-back 30', opt(E21, lookback=30), f'{B5}/e27_lookback30.json')

In [ ]:
run('E28 input masking 0.15', opt(E21, chan_mask=0.15), f'{B5}/e28_chan_mask.json')

In [ ]:
run('E06 context length U[20, 60]', opt(E21, context_min=20), f'{B5}/e06_context_mask.json')

In [ ]:
# XGBoost with the E26 inputs (reference; about 3 min)
if not os.path.exists(f'{B5}/xgb_e26.json'):
    !python scripts/baselines_tabular.py --extra-features e03_e26 --out "$B5/xgb_e26.json"
x = json.load(open(f'{B5}/xgb_e26.json'))['results']
print({h: x[h]['model']['xgb_trend']['macro_f1'] for h in ('h1', 'h5', 'h10')})

In [ ]:
# Summary against E21 (per-horizon best epoch) -> batch5_summary.json (read by batch 7)
!python scripts/compare_batch.py --per-horizon-best --control-label E21 --control "$E21_JSON" \
    --run E26 "$B5/e26_vol_inputs.json" --run E27 "$B5/e27_lookback30.json" --run E28 "$B5/e28_chan_mask.json" \
    --run E06 "$B5/e06_context_mask.json" --out "$B5/batch5_summary.json"